In [1]:
#!pip install scipy 

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [3]:
# Paths
train_dir = "dataset/train"
val_dir = "dataset/val"
test_dir = "dataset/test"

In [4]:
# Image size and batch size
img_height = 224
img_width = 224
batch_size = 32

In [5]:
# 1. Data augmentation for training data
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rescale=1.0/255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

In [6]:
# 2. Only preprocess_input for validation and test
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [7]:
# 3. Load training data
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary'
)

Found 40 images belonging to 2 classes.


In [8]:
# 4. Load validation data
val_data = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

Found 10 images belonging to 2 classes.


In [9]:
# 5. Load test data
test_data = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

Found 20 images belonging to 2 classes.


In [10]:
# 6. Print class labels
print("Class indices:", train_data.class_indices)

Class indices: {'normal': 0, 'pneumonia': 1}


In [11]:
# 7. Check one batch
images, labels = next(train_data)

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels)

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Labels: [1. 0. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 0. 0. 0. 1. 0. 0. 1. 1. 0. 0.
 0. 0. 0. 0. 1. 0. 1. 0.]


In [12]:
# =========================
# 5. Load pretrained model
# =========================
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(img_height, img_width, 3)
)

In [13]:
# Freeze the pretrained layers
base_model.trainable = False

In [14]:
# =========================
# 6. Build transfer learning model
# =========================
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

In [15]:
# =========================
# 7. Compile model
# =========================
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [16]:
# =========================
# 8. Show model summary
# =========================
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [17]:
# =========================
# 9. Train model
# =========================
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.4500 - loss: 1.1368 - val_accuracy: 0.6000 - val_loss: 0.7520
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4250 - loss: 0.9607 - val_accuracy: 0.5000 - val_loss: 0.7320
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5750 - loss: 0.7840 - val_accuracy: 0.5000 - val_loss: 0.7731
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 448ms/step - accuracy: 0.5250 - loss: 0.9451 - val_accuracy: 0.3000 - val_loss: 0.7345
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4500 - loss: 0.7913 - val_accuracy: 0.5000 - val_loss: 0.7812
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5500 - loss: 0.7971 - val_accuracy: 0.5000 - val_loss: 0.8482
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 485ms/step - accuracy: 0.5000 - loss: 0.8898 - val_accuracy: 0.5000 - val_loss: 0.8498
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 350ms/step - accuracy: 0.4000 - loss: 0.8599 - val_accuracy: 0.5000 - val_loss: 0.8220
Epoch 9

In [18]:
# =========================
# 10. Evaluate on test data
# =========================
test_loss, test_acc = model.evaluate(test_data)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 389ms/step - accuracy: 0.2500 - loss: 0.8546
Test Loss: 0.8546259999275208
Test Accuracy: 0.25
